In [0]:
import random
from datetime import date
from pyspark.sql import Row

random.seed(42)   # same "random" results every run, so we can compare
base_path = "/Volumes/workspace/patient_notes/raw_notes"

# standard name (answer key)  ->  ways a doctor might write it (note text)
condition_forms = {
    "pneumonia":               ["pneumonia", "Pneumonia", "PNA"],
    "type 2 diabetes":         ["type 2 diabetes", "T2DM", "DM2", "diabetes mellitus type 2"],
    "hypertension":            ["hypertension", "HTN", "high blood pressure"],
    "asthma":                  ["asthma", "Asthma"],
    "lung tumor":              ["lung tumor", "lung mass", "pulmonary neoplasm"],
    "chronic kidney disease":  ["chronic kidney disease", "CKD"],
    "malaria":                 ["malaria", "Malaria"],
    "urinary tract infection": ["urinary tract infection", "UTI"],
    "migraine":                ["migraine", "Migraine"],
    "copd":                    ["COPD", "copd", "chronic obstructive pulmonary disease"],
    "fatty liver":             ["fatty liver", "hepatic steatosis", "NAFLD"],
}
conditions = list(condition_forms)   # the standard names

# How doctors phrase each status
phrases = {
    "current":   ["presents with {c}", "diagnosed with {c}", "currently being treated for {c}"],
    "history":   ["known case of {c}", "history of {c}", "past medical history significant for {c}"],
    "ruled_out": ["{c} ruled out", "negative for {c}", "no evidence of {c}"],
}
complaints = ["fever and cough", "shortness of breath", "headache", "fatigue", "chest pain", "abdominal pain","digestive issues"]
note_types = ["progress note", "consult note", "discharge summary"]
patients   = [f"P{i:04d}" for i in range(1, 21)]   # P0001 ... P0020

In [0]:
def make_note(note_id, patient_id, note_date):
    chosen = random.sample(conditions, k=random.randint(1, 3))   # 1-3 conditions per note
    labels, sentences = [], []
    for c in chosen:
        status  = random.choice(list(phrases))
        written = random.choice(condition_forms[c])          # messy form goes in the text
        s = random.choice(phrases[status]).format(c=written)
        sentences.append(s[0].upper() + s[1:] + ".")         # uppercase first letter only
        labels.append(Row(condition=c, status=status))       # standard name goes in the answer key
    text = (f"Chief complaint: {random.choice(complaints)}.\n"
            f"Assessment: {' '.join(sentences)}\n"
            f"Plan: follow up in {random.randint(1, 4)} weeks.")
    note = Row(note_id=note_id, patient_id=patient_id, note_date=str(note_date),
               note_type=random.choice(note_types), note_text=text)
    return note, labels

In [0]:
day = date(2026, 11, 1)
notes, truth = [], []

for n in range(60):
    note_id = f"N{day:%Y%m%d}-{n:03d}"
    note, labels = make_note(note_id, random.choice(patients), day)
    notes.append(note)
    truth.append(Row(note_id=note_id, labels=labels))

notes_df = spark.createDataFrame(notes)    # list of Rows -> Spark DataFrame
truth_df = spark.createDataFrame(truth)

notes_df.write.mode("overwrite").json(f"{base_path}/incoming/{day}")       # what the pipeline sees
truth_df.write.mode("overwrite").json(f"{base_path}/ground_truth/{day}")   # the hidden answer key

display(notes_df.limit(5))
display(dbutils.fs.ls(f"{base_path}/incoming/{day}"))

In [0]:
notes_path = f"/Volumes/workspace/patient_notes/raw_notes/incoming/2026-11-01"
truth_path = f"/Volumes/workspace/patient_notes/raw_notes/ground_truth/2026-11-01"

notes_back = spark.read.json(notes_path)   # Spark reads the whole folder; marker files are ignored
truth_back = spark.read.json(truth_path)

print("Notes:", notes_back.count())        # should be 60
notes_back.printSchema()                   # column names and types
display(notes_back.limit(5))

In [0]:
display(
    notes_back.join(truth_back, on="note_id")      # match each note to its answer key
              .select("note_id", "patient_id", "note_text", "labels")
              .limit(5)
)